In [9]:
import random

# Grammar rules
rules = {
    "S": [["M", "M"], ["B", "M", "A"], ["P", "M", "R"]],
    "M": [["M", "M"], ["B", "M", "A"], ["P", "M", "R"], ["C"]],
    "B": [["clone"], ["group"]],
    "A": [["add"], ["concat"], ["matmul"]],
    "P": [["im2col"], ["permute"], ["identity"]],
    "R": [["col2im"], ["permute"], ["identity"]],
    "C": [["linear"], ["relu"], ["norm"], ["softmax"], ["identity"]],
}

# Terminals = real operations
terminals = {
    "clone","group","add","concat","matmul",
    "im2col","permute","identity","col2im",
    "linear","relu","norm","softmax"
}

def is_terminal(sym):
    return sym in terminals

def expand_once(structure):
    terminalProb = 0.32  # correct value

    for i, sym in enumerate(structure):
        if sym in rules:

            if sym == "M":
                expansion = random.choices(
                    [["M","M"], ["B","M","A"], ["P","M","R"], ["C"]],
                    weights=[(1-terminalProb)/3, (1-terminalProb)/3, (1-terminalProb)/3, terminalProb]
                )[0]
            else:
                expansion = random.choice(rules[sym])

            # ✅ RETURN MUST BE INSIDE LOOP
            return structure[:i] + expansion + structure[i+1:], sym, expansion

    return structure, None, None


def sample_with_steps():
    structure = ["S"]
    step = 0

    print(f"Step {step}: {' '.join(structure)}")

    while True:
        new_structure, expanded_sym, expansion = expand_once(structure)

        if expanded_sym is None:
            break

        step += 1
        print(f"\nStep {step}: expand {expanded_sym} → {' '.join(expansion)}")
        print(f"Result: {' '.join(new_structure)}")

        structure = new_structure

    print("\nFinal architecture:")
    print(" → ".join(structure))

    return structure   # ✅ ADD THIS





def explain_structure(structure):
    # Very simple interpretation based on patterns
    print("Explaining the Structure")
    if "clone" in structure and ("add" in structure or "concat" in structure):
        merge_op = "add" if "add" in structure else "concat"
        print("\nInterpretation:")
        print("This is a branching structure:")
        print(" - Input is split using 'clone'")
        print(" - Each branch is processed (here mostly identity/linear etc.)")
        print(f" - Branches are merged using '{merge_op}'")

        # show pseudo forward pass
        print("\nPseudo forward pass:")
        print("x1 = identity(x)")
        print("x2 = identity(x)")
        print(f"out = {merge_op}(x1, x2)")

    elif "im2col" in structure and "col2im" in structure:
        print("\nInterpretation:")
        print("This is a convolution-like block:")
        print(" - im2col extracts patches")
        print(" - linear applies filters")
        print(" - col2im restores spatial structure")

        print("\nPseudo forward pass:")
        print("patches = im2col(x)")
        print("features = linear(patches)")
        print("out = col2im(features)")

    else:
        print("\nInterpretation:")
        print("This is a simple sequential model:")
        print(" → ".join(structure))

In [10]:
# Run once
structure = sample_with_steps()   # ✅ capture output
explain_structure(structure)      # ✅ call explanation

Step 0: S

Step 1: expand S → M M
Result: M M

Step 2: expand M → P M R
Result: P M R M

Step 3: expand P → im2col
Result: im2col M R M

Step 4: expand M → M M
Result: im2col M M R M

Step 5: expand M → M M
Result: im2col M M M R M

Step 6: expand M → C
Result: im2col C M M R M

Step 7: expand C → linear
Result: im2col linear M M R M

Step 8: expand M → M M
Result: im2col linear M M M R M

Step 9: expand M → C
Result: im2col linear C M M R M

Step 10: expand C → identity
Result: im2col linear identity M M R M

Step 11: expand M → C
Result: im2col linear identity C M R M

Step 12: expand C → linear
Result: im2col linear identity linear M R M

Step 13: expand M → C
Result: im2col linear identity linear C R M

Step 14: expand C → norm
Result: im2col linear identity linear norm R M

Step 15: expand R → identity
Result: im2col linear identity linear norm identity M

Step 16: expand M → P M R
Result: im2col linear identity linear norm identity P M R

Step 17: expand P → permute
Result: im2co